# Three Ways to Rewrite a Photograph -- submission walkthrough

End-to-end demo of our three editing modules (object replacement, relocation, and style transfer) run on a single hand-picked photo. Designed for a Colab A100. Each cell saves its output inline.

Authors: Stefan Arni Arnarsson, Aryaan Verma, Chien-Wei Wang.

## 1. Setup
Clone the repo and install requirements. Run once per session.

In [ ]:
# Silence the noisy stdout/stderr from pip, HF, transformers, diffusers, etc.
import os, warnings, logging
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
os.environ['DIFFUSERS_VERBOSITY'] = 'error'
warnings.filterwarnings('ignore')
for name in ('transformers', 'diffusers', 'huggingface_hub', 'torchvision', 'open_clip', 'urllib3', 'PIL'):
    logging.getLogger(name).setLevel(logging.ERROR)

from pathlib import Path
import os, subprocess, sys

REPO = Path('/content/repo')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/stefan-arni/cs5788_final.git', str(REPO)], check=True)
os.chdir(REPO)

%pip install -q -r requirements.txt
if (REPO / 'relocate' / 'requirements.txt').exists():
    %pip install -q -r relocate/requirements.txt
if (REPO / 'style' / 'requirements.txt').exists():
    %pip install -q -r style/requirements.txt

for sub in ['replace', 'relocate', 'style']:
    p = str(REPO / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import torch
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only (will be very slow)')

## 2. Load the demo photo
One hand-picked photo (`data/demo/headline.jpg`). The rigorous quantitative ablation across 20 COCO photos lives in `scripts/run_ablation.py`.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

DEMO_IMAGE = 'data/demo/headline.jpg'
src_path = Path(DEMO_IMAGE)
if not src_path.exists():
    raise FileNotFoundError(f'demo image missing at {DEMO_IMAGE}')

source = Image.open(src_path).convert('RGB').resize((512, 512))
print(f'demo photo: {src_path.name}  ({source.size[0]}x{source.size[1]})')
source

## 3. Module 1 -- Object Replacement
Linear-decay attention swap schedule + pixel composite over an attention-derived mask. Prints reconstruction LPIPS, CLIP-directional, and background LPIPS.

In [ ]:
import torch
from sd_components import load_sd, decode_latents, encode_prompt
from null_text_inv import null_text_inversion, sample_with_null
from editor import Editor, _to_pil
from schedules import linear_decay_replaced
from metrics import reconstruction_lpips, clip_directional_similarity, background_lpips

c = load_sd()
editor = Editor(c)

src_prompt = 'a photograph of a golden retriever sitting on grass'
tgt_prompt = 'a photograph of a red fox sitting on grass'
print(f'source: {src_prompt!r}')
print(f'target: {tgt_prompt!r}')

nt = null_text_inversion(c, source, src_prompt, num_inference_steps=50, guidance_scale=7.5, inner_steps=10, verbose=False)
z0 = sample_with_null(c, encode_prompt(c, src_prompt), nt.null_embeds, nt.z_T, 50, 7.5)
recon = _to_pil(decode_latents(c, z0).clamp(-1, 1))
recon_lpips = reconstruction_lpips(source, recon)

replaced = editor.edit(source, src_prompt, tgt_prompt, schedule=linear_decay_replaced(), mask_mode='attention', composite_mode='strict')
mask = editor.derive_mask(source, src_prompt, tgt_prompt)
clip_dir = clip_directional_similarity(source, replaced, src_prompt, tgt_prompt)
bg = background_lpips(source, replaced, mask)

print(f'\nrecon LPIPS:      {recon_lpips:.4f}  (target < 0.05)')
print(f'CLIP-directional: {clip_dir:.4f}  (target > 0.20)')
print(f'background LPIPS: {bg:.4f}  (target < 0.10)')

fig, ax = plt.subplots(1, 2, figsize=(7, 3.5))
ax[0].imshow(source);   ax[0].set_title('source'); ax[0].axis('off')
ax[1].imshow(replaced); ax[1].set_title(f'after replacement (CLIP-dir={clip_dir:.2f})'); ax[1].axis('off')
plt.tight_layout(); plt.show()

## 4. Module 2 -- Object Relocation
Move the freshly-replaced subject. The pipeline takes a source mask (where the object IS) and a target mask (where it SHOULD GO). Hard-coded rectangles for reproducibility.

In [ ]:
import numpy as np
from PIL import ImageDraw
from pipeline.relocation_pipeline import ObjectRelocationPipeline

W, H = replaced.size
src_mask = Image.new('L', (W, H), 0)
tgt_mask = Image.new('L', (W, H), 0)
ImageDraw.Draw(src_mask).rectangle([int(W*0.15), int(H*0.30), int(W*0.55), int(H*0.90)], fill=255)
ImageDraw.Draw(tgt_mask).rectangle([int(W*0.50), int(H*0.30), int(W*0.90), int(H*0.90)], fill=255)

relocation_pipe = ObjectRelocationPipeline()
relocated, _ = relocation_pipe(replaced, tgt_prompt, src_mask, tgt_mask,
    use_noise_shift=True, seed=42, num_inference_steps=30,
    sdedit_strength=0.6, guidance_scale=7.5)

src_arr = np.asarray(replaced).astype(float)
src_norm = np.asarray(src_mask, dtype=float) / 255.0
tgt_norm = np.asarray(tgt_mask, dtype=float) / 255.0
GREEN, RED, ALPHA = np.array([60, 200, 100]), np.array([220, 60, 60]), 0.45
overlay = src_arr.copy()
overlay = overlay * (1 - ALPHA * src_norm[..., None]) + GREEN * (ALPHA * src_norm[..., None])
overlay = overlay * (1 - ALPHA * tgt_norm[..., None]) + RED   * (ALPHA * tgt_norm[..., None])
overlay = overlay.clip(0, 255).astype('uint8')

fig, ax = plt.subplots(1, 3, figsize=(10, 3.5))
ax[0].imshow(replaced);  ax[0].set_title('input (after replacement)'); ax[0].axis('off')
ax[1].imshow(overlay);   ax[1].set_title('source (green) -> target (red)'); ax[1].axis('off')
ax[2].imshow(relocated); ax[2].set_title('after relocation'); ax[2].axis('off')
plt.tight_layout(); plt.show()

## 5. Module 3 -- Style Transfer
Repaint the relocated image in Van Gogh via a per-style LoRA adapter trained on WikiArt.

In [ ]:
import inference, styles
from inference import set_style_mix, build_prompt, stylize_image

inference.MODEL_ID = 'stable-diffusion-v1-5/stable-diffusion-v1-5'
style_root = REPO / 'style'
for k, cfg in styles.STYLES.items():
    cfg['lora_dir'] = str(style_root / cfg['lora_dir'])

style_pipe = inference.load_pipeline(inference.get_device())
style_keys = [('van_gogh', 1.0)]
set_style_mix(style_pipe, style_keys)
styled = stylize_image(style_pipe, relocated, build_prompt(style_keys),
    strength=0.4, guidance_scale=7.5, num_inference_steps=40, seed=42)

fig, ax = plt.subplots(1, 2, figsize=(7, 3.5))
ax[0].imshow(relocated); ax[0].set_title('input (after relocation)'); ax[0].axis('off')
ax[1].imshow(styled);    ax[1].set_title('after Van Gogh stylization'); ax[1].axis('off')
plt.tight_layout(); plt.show()

## 6. Cascade view
Source -> replace -> relocate -> stylize, all from one upload.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for ax, img, title in zip(
    axes,
    [source, replaced, relocated, styled],
    ['source', '1. replaced', '2. relocated', '3. stylized'],
):
    ax.imshow(img); ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()

## 7. Live demo via Gradio
Launches the unified web UI. Prints a public `*.gradio.live` URL. Blocks while the app runs; interrupt to stop.

In [ ]:
!python platform/app.py